# Train TRRUST Classifiers

Pipeline:

1. **Load** gene embeddings + TRRUST data.
2. **(Optional)** Restrict training data to a gene subset from a file.
3. **(Optional)** Write the current gene universe to a file for cross-model comparisons.
4. **Hyperparameter sweep with 5-fold CV per config** — 5 LRs × 4 epoch counts
   for binary + ternary. For each config we run stratified 5-fold CV and save
   per-fold train/test loss curves, per-fold classification reports, a summary
   CSV with mean/std accuracy and macro F1, and a heatmap of **mean macro F1**
   across LR × epochs under `REPORTS_DIR`.
5. **Final results** — re-run 5-fold CV at the chosen LR / epoch count for
   binary + ternary (inspect the sweep heatmap, then set `*_LR` / `*_EPOCHS`).
6. **Save final results** as a pickle for downstream analysis.

This notebook is model-agnostic — point `EMBEDDINGS_PATH` at any h5ad file produced
by `src/scgpt/encode.py` or `src/geneformer/encode.py`.


In [1]:
import itertools
import pickle
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report

from src.trrust import (
    BINARY_LABEL_NAMES,
    BINARY_LABELS,
    TERNARY_LABEL_NAMES,
    TERNARY_LABELS,
    count_relationship_pairs,
    cross_validate,
    filter_data_by_genes,
    generate_shared_none_pairs,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
)


## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models, and point `REPORTS_DIR` /
`CV_RESULTS_PATH` at model-specific output locations. Set `GENE_SUBSET_PATH` to
restrict training to pairs where both TF and target appear in a gene-list file
(see `data/scgpt_binary_genes.txt`).

`LR_GRID` and `EPOCH_GRID` define the hyperparameter sweep; each `(lr, epochs)`
config is evaluated with 5-fold stratified CV. After inspecting the mean macro
F1 heatmap, fill in `BINARY_LR` / `BINARY_EPOCHS` / `TERNARY_LR` /
`TERNARY_EPOCHS` before running the final-results cells.


In [2]:
EMBEDDINGS_PATH = Path("../data/embeddings/geneformer_nkt.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
REPORTS_DIR = Path("../reports/geneformer/hp_tuning_nkt")
CV_RESULTS_PATH = Path("../data/results/geneformer_cv_results_nkt.pkl")
GENE_SUBSET_PATH: Path | None = None  # e.g. Path("../data/scgpt_binary_genes.txt") or None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

LR_GRID = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
EPOCH_GRID = [8, 12, 20, 50]


print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"CV results: {CV_RESULTS_PATH}")
print(f"Gene subset: {GENE_SUBSET_PATH}")


Device: cuda
Embeddings: ../data/embeddings/geneformer_nkt.h5ad
TRRUST: ../data/trrust_rawdata.human.tsv
Reports dir: ../reports/geneformer/hp_tuning_nkt
CV results: ../data/results/geneformer_cv_results_nkt.pkl
Gene subset: None


## Load embeddings and TRRUST data

In [3]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

# Shared vocabulary across scFMs for this cell type. All training records
# (relationship + None) are restricted to this intersection so every scFM sees
# the same (tf, target) samples — cross-model agreement analysis then compares
# predictions on an identical pair set.
SHARED_SCFMS = ["scgpt", "geneformer"]
_scfm, _cell_tag = EMBEDDINGS_PATH.stem.split("_", 1)
scfm_vocabs = [
    set(load_gene_embeddings(EMBEDDINGS_PATH.parent / f"{name}_{_cell_tag}.h5ad").keys())
    for name in SHARED_SCFMS
]
shared_vocab = set.intersection(*scfm_vocabs)

n_binary_none = count_relationship_pairs(TRRUST_PATH, shared_vocab)
n_ternary_none = count_relationship_pairs(TRRUST_PATH, shared_vocab, exclude_unknown=True) // 2
binary_none_pairs = generate_shared_none_pairs(
    TRRUST_PATH, scfm_vocabs, n=n_binary_none, seed=42,
)
ternary_none_pairs = generate_shared_none_pairs(
    TRRUST_PATH, scfm_vocabs, n=n_ternary_none, seed=42,
)

binary_data = load_binary_trrust_data(
    TRRUST_PATH, gene_embeddings, none_pairs=binary_none_pairs,
)
ternary_data = load_ternary_trrust_data(
    TRRUST_PATH, gene_embeddings, none_pairs=ternary_none_pairs,
)

# Restrict relationship records to the shared vocab (None records are already
# drawn from the intersection, so they're unaffected).
binary_data = filter_data_by_genes(binary_data, shared_vocab)
ternary_data = filter_data_by_genes(ternary_data, shared_vocab)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Shared vocab (intersection of {SHARED_SCFMS}): {len(shared_vocab)} genes")
print(f"Binary samples: {len(binary_data.records)} ({n_binary_none} shared None)")
print(f"Ternary samples: {len(ternary_data.records)} ({n_ternary_none} shared None)")


Gene embeddings: 11495 genes, 1152d
Shared vocab (intersection of ['scgpt', 'geneformer']): 10613 genes
Binary samples: 7882 (3941 shared None)
Ternary samples: 3055 (1018 shared None)


## (Optional) Restrict training data to a gene subset

If `GENE_SUBSET_PATH` is set, keep only TRRUST records where both TF and target are
listed in the file (one gene symbol per line). A no-op when `GENE_SUBSET_PATH is None`.


In [5]:
if GENE_SUBSET_PATH is not None:
    subset_genes = {
        line.strip()
        for line in Path(GENE_SUBSET_PATH).read_text().splitlines()
        if line.strip()
    }
    print(f"Subset file: {GENE_SUBSET_PATH} ({len(subset_genes)} genes)")

    before_binary = len(binary_data.records)
    before_ternary = len(ternary_data.records)
    binary_data = filter_data_by_genes(binary_data, subset_genes)
    ternary_data = filter_data_by_genes(ternary_data, subset_genes)
    print(f"Binary records: {before_binary} -> {len(binary_data.records)}")
    print(f"Ternary records: {before_ternary} -> {len(ternary_data.records)}")
else:
    print("GENE_SUBSET_PATH is None — using all records.")


Subset file: ../data/scgpt_binary_genes_nkt.txt (1782 genes)
Binary records: 9030 -> 7822
Ternary records: 3516 -> 3055


## (Optional) Save current gene lists for future cross-model runs

Write the union of TFs + targets from the current `binary_data` / `ternary_data` to
two `.txt` files so another run (e.g. a different embedding model) can be restricted
to the same gene universe via `GENE_SUBSET_PATH` above. Edit the output paths for
the model you are running.


In [8]:
BINARY_GENE_FILE = Path("../data/scgpt_binary_genes_nkt.txt")
TERNARY_GENE_FILE = Path("../data/scgpt_ternary_genes_nkt.txt")

binary_genes = sorted({g for r in binary_data.records for g in (r.tf, r.target)})
BINARY_GENE_FILE.write_text("\n".join(binary_genes) + "\n")
print(f"Saved {len(binary_genes)} binary genes to {BINARY_GENE_FILE}")

ternary_genes = sorted({g for r in ternary_data.records for g in (r.tf, r.target)})
TERNARY_GENE_FILE.write_text("\n".join(ternary_genes) + "\n")
print(f"Saved {len(ternary_genes)} ternary genes to {TERNARY_GENE_FILE}")


Saved 1802 binary genes to ../data/scgpt_binary_genes_nkt.txt
Saved 1590 ternary genes to ../data/scgpt_ternary_genes_nkt.txt


## Hyperparameter sweep helpers

Each config runs 5-fold stratified cross-validation (seed=42) so rankings are
robust to any single split. For every `(lr, epochs)` combination we save a
single plot with overlaid per-fold train/test loss curves, a text report with
per-fold classification reports plus mean ± std accuracy / macro F1, and a row
in `summary.csv`. After the loop, a heatmap of **mean macro F1** over LR ×
epochs is saved alongside the per-config outputs.


In [4]:
def _save_loss_plot(cv_result, lr, epochs, title_prefix, out_path):
    plt.figure(figsize=(8, 4))
    xs = range(1, epochs + 1)
    colors = plt.get_cmap("tab10").colors
    for i, fold in enumerate(cv_result.per_fold):
        color = colors[i % len(colors)]
        plt.plot(xs, fold.train_losses, color=color, linestyle="-",
                 label=f"Fold {i + 1} train")
        plt.plot(xs, fold.test_losses, color=color, linestyle="--",
                 label=f"Fold {i + 1} test")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} — lr={lr:.0e}, epochs={epochs}")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def _fold_macro_f1s(cv_result):
    return np.array([
        fold.classification_report["macro avg"]["f1-score"]
        for fold in cv_result.per_fold
    ])


def _save_text_report(cv_result, label_names, out_path):
    target_names = [label_names[i] for i in range(len(label_names))]
    name_to_label = {name: idx for idx, name in label_names.items()}
    lines = []
    for i, fold in enumerate(cv_result.per_fold):
        preds = fold.gene_predictions
        true_ids = preds["true_relationship"].map(name_to_label).to_numpy()
        pred_ids = preds["predicted_relationship"].map(name_to_label).to_numpy()
        lines.append(f"=== Fold {i + 1} ===")
        lines.append(classification_report(
            true_ids, pred_ids, target_names=target_names, zero_division=0
        ))
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    lines.append(f"Mean accuracy: {accs.mean():.3f} ± {accs.std():.3f}")
    lines.append(f"Mean macro F1: {f1s.mean():.3f} ± {f1s.std():.3f}")
    out_path.write_text("\n".join(lines))


def _summary_row(lr, epochs, cv_result, label_names):
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    row = {
        "lr": lr,
        "epochs": epochs,
        "mean_accuracy": accs.mean(),
        "std_accuracy": accs.std(),
        "mean_macro_f1": f1s.mean(),
        "std_macro_f1": f1s.std(),
    }
    agg = cv_result.aggregate_classification_report
    row["pooled_accuracy"] = agg["accuracy"]
    for name in ("macro avg", "weighted avg"):
        for metric in ("precision", "recall", "f1-score"):
            row[f"pooled_{name}_{metric}"] = agg[name][metric]
    for i in range(len(label_names)):
        name = label_names[i]
        row[f"pooled_{name}_precision"] = agg[name]["precision"]
        row[f"pooled_{name}_recall"] = agg[name]["recall"]
        row[f"pooled_{name}_f1"] = agg[name]["f1-score"]
    return row


def _save_f1_heatmap(summary_df, title, out_path):
    pivot = summary_df.pivot(
        index="lr", columns="epochs", values="mean_macro_f1"
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.0e}" for v in pivot.index])
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Learning rate")
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    fig.colorbar(im, ax=ax, label="mean macro F1")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def run_hp_sweep(
    data,
    *,
    label_map,
    label_names,
    use_class_weights,
    out_dir,
    title_prefix,
    n_splits=5,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for lr, epochs in itertools.product(LR_GRID, EPOCH_GRID):
        cv_result = cross_validate(
            data,
            embsize=embsize,
            label_map=label_map,
            lr=lr,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            use_class_weights=use_class_weights,
            n_splits=n_splits,
            device=DEVICE,
            seed=42,
        )
        tag = f"lr{lr:.0e}_e{epochs}"
        _save_loss_plot(cv_result, lr, epochs, title_prefix, out_dir / f"loss_{tag}.png")
        _save_text_report(cv_result, label_names, out_dir / f"report_{tag}.txt")
        row = _summary_row(lr, epochs, cv_result, label_names)
        rows.append(row)
        print(f"  {tag}: mean acc={row['mean_accuracy']:.3f} ± {row['std_accuracy']:.3f}, "
              f"mean macro F1={row['mean_macro_f1']:.3f} ± {row['std_macro_f1']:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    _save_f1_heatmap(
        summary,
        f"{title_prefix} mean macro F1 (5-fold CV)",
        out_dir / "f1_heatmap.png",
    )
    return summary


## Hyperparameter sweep — binary classifier

20 configs (5 LRs × 4 epoch counts), each run through 5-fold stratified CV.
No class weights (binary data is balanced by construction). Outputs land in
`REPORTS_DIR/binary/`.


In [5]:
binary_summary = run_hp_sweep(
    binary_data,
    label_map=BINARY_LABELS,
    label_names=BINARY_LABEL_NAMES,
    use_class_weights=False,
    out_dir=REPORTS_DIR / "binary",
    title_prefix="Binary",
)
binary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.734 ± 0.010, mean macro F1=0.734 ± 0.010
  lr1e-02_e12: mean acc=0.733 ± 0.012, mean macro F1=0.732 ± 0.012
  lr1e-02_e20: mean acc=0.727 ± 0.014, mean macro F1=0.726 ± 0.014
  lr1e-02_e50: mean acc=0.735 ± 0.006, mean macro F1=0.734 ± 0.006
  lr1e-03_e8: mean acc=0.747 ± 0.012, mean macro F1=0.747 ± 0.012
  lr1e-03_e12: mean acc=0.743 ± 0.016, mean macro F1=0.743 ± 0.016
  lr1e-03_e20: mean acc=0.744 ± 0.014, mean macro F1=0.744 ± 0.014
  lr1e-03_e50: mean acc=0.740 ± 0.017, mean macro F1=0.740 ± 0.017
  lr1e-04_e8: mean acc=0.732 ± 0.007, mean macro F1=0.732 ± 0.007
  lr1e-04_e12: mean acc=0.738 ± 0.004, mean macro F1=0.738 ± 0.005
  lr1e-04_e20: mean acc=0.742 ± 0.003, mean macro F1=0.742 ± 0.003
  lr1e-04_e50: mean acc=0.743 ± 0.007, mean macro F1=0.743 ± 0.007
  lr1e-05_e8: mean acc=0.648 ± 0.011, mean macro F1=0.648 ± 0.011
  lr1e-05_e12: mean acc=0.666 ± 0.006, mean macro F1=0.666 ± 0.006
  lr1e-05_e20: mean acc=0.690 ± 0.005, mean macro F1=0.690 ± 0.005

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,pooled_weighted avg_precision,pooled_weighted avg_recall,pooled_weighted avg_f1-score,pooled_None_precision,pooled_None_recall,pooled_None_f1,pooled_Relationship_precision,pooled_Relationship_recall,pooled_Relationship_f1
4,0.0010,8,0.746635,0.011584,0.746526,0.011692,0.746638,0.746781,0.746638,0.746601,0.746781,0.746638,0.746601,0.752730,0.734585,0.743547,0.740833,0.758691,0.749655
6,0.0010,20,0.744480,0.014215,0.744285,0.014380,0.744481,0.744716,0.744481,0.744420,0.744716,0.744481,0.744420,0.752291,0.729003,0.740464,0.737140,0.759959,0.748376
11,0.0001,50,0.742959,0.007102,0.742932,0.007135,0.742959,0.742995,0.742959,0.742949,0.742995,0.742959,0.742949,0.745954,0.736869,0.741384,0.740035,0.749048,0.744515
5,0.0010,12,0.742703,0.016341,0.742572,0.016420,0.742705,0.742945,0.742705,0.742641,0.742945,0.742705,0.742641,0.750589,0.726973,0.738592,0.735301,0.758437,0.746690
10,0.0001,20,0.741690,0.003172,0.741633,0.003225,0.741690,0.741750,0.741690,0.741674,0.741750,0.741690,0.741674,0.745553,0.733824,0.739642,0.737947,0.749556,0.743706


## Hyperparameter sweep — ternary classifier

20 configs, each run through 5-fold stratified CV with inverse-frequency class
weights (ternary data is imbalanced). Outputs land in `REPORTS_DIR/ternary/`.


In [6]:
ternary_summary = run_hp_sweep(
    ternary_data,
    label_map=TERNARY_LABELS,
    label_names=TERNARY_LABEL_NAMES,
    use_class_weights=True,
    out_dir=REPORTS_DIR / "ternary",
    title_prefix="Ternary",
)
ternary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.522 ± 0.020, mean macro F1=0.514 ± 0.017
  lr1e-02_e12: mean acc=0.524 ± 0.021, mean macro F1=0.517 ± 0.019
  lr1e-02_e20: mean acc=0.532 ± 0.016, mean macro F1=0.524 ± 0.014
  lr1e-02_e50: mean acc=0.526 ± 0.022, mean macro F1=0.519 ± 0.021
  lr1e-03_e8: mean acc=0.534 ± 0.011, mean macro F1=0.527 ± 0.008
  lr1e-03_e12: mean acc=0.535 ± 0.011, mean macro F1=0.528 ± 0.008
  lr1e-03_e20: mean acc=0.532 ± 0.009, mean macro F1=0.525 ± 0.007
  lr1e-03_e50: mean acc=0.531 ± 0.018, mean macro F1=0.523 ± 0.017
  lr1e-04_e8: mean acc=0.490 ± 0.013, mean macro F1=0.481 ± 0.011
  lr1e-04_e12: mean acc=0.499 ± 0.014, mean macro F1=0.490 ± 0.010
  lr1e-04_e20: mean acc=0.506 ± 0.016, mean macro F1=0.497 ± 0.012
  lr1e-04_e50: mean acc=0.523 ± 0.018, mean macro F1=0.514 ± 0.016
  lr1e-05_e8: mean acc=0.411 ± 0.014, mean macro F1=0.403 ± 0.013
  lr1e-05_e12: mean acc=0.422 ± 0.014, mean macro F1=0.414 ± 0.013
  lr1e-05_e20: mean acc=0.440 ± 0.014, mean macro F1=0.431 ± 0.012

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,...,pooled_weighted avg_f1-score,pooled_Activation_precision,pooled_Activation_recall,pooled_Activation_f1,pooled_Repression_precision,pooled_Repression_recall,pooled_Repression_f1,pooled_None_precision,pooled_None_recall,pooled_None_f1
5,0.001,12,0.534861,0.010576,0.527585,0.008018,0.534861,0.528835,0.527235,0.527830,...,0.535925,0.559475,0.564570,0.562011,0.421659,0.441496,0.431349,0.605372,0.575639,0.590131
4,0.001,8,0.533552,0.010707,0.526577,0.008458,0.533552,0.527959,0.526177,0.526807,...,0.534748,0.557834,0.562914,0.560363,0.421053,0.443908,0.432179,0.604990,0.571709,0.587879
6,0.001,20,0.531915,0.009430,0.524832,0.007455,0.531915,0.525622,0.524751,0.525065,...,0.532793,0.557119,0.557119,0.557119,0.423611,0.441496,0.432369,0.596134,0.575639,0.585707
2,0.010,20,0.532242,0.016016,0.524216,0.013986,0.532242,0.524338,0.524911,0.524568,...,0.532685,0.567912,0.553808,0.560771,0.424207,0.435464,0.429762,0.580897,0.585462,0.583170
7,0.001,50,0.530606,0.017791,0.523462,0.017010,0.530606,0.524114,0.523406,0.523676,...,0.531426,0.552326,0.550497,0.551410,0.418605,0.434258,0.426288,0.601413,0.585462,0.593330


## Final results — binary classifier

Re-run 5-fold CV at the chosen `(BINARY_LR, BINARY_EPOCHS)` (read off the sweep
heatmap) to produce the final reportable per-fold accuracies and the pooled
(aggregate) classification report.


In [7]:
BINARY_LR = 1e-3
BINARY_EPOCHS = 8
BINARY_FOLDS = 5

In [8]:
binary_cv = cross_validate(
    binary_data,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    n_splits=BINARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(binary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(binary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.753
Fold 2: accuracy=0.763
Fold 3: accuracy=0.732
Fold 4: accuracy=0.735
Fold 5: accuracy=0.750

              precision  recall  f1-score   support
None              0.753   0.735     0.744  3941.000
Relationship      0.741   0.759     0.750  3941.000
accuracy          0.747   0.747     0.747     0.747
macro avg         0.747   0.747     0.747  7882.000
weighted avg      0.747   0.747     0.747  7882.000


## Final results — ternary classifier

Re-run 5-fold CV at the chosen `(TERNARY_LR, TERNARY_EPOCHS)` with inverse-
frequency class weights to produce the final reportable per-fold accuracies and
pooled classification report.


In [9]:
TERNARY_LR = 1e-3
TERNARY_EPOCHS = 12
TERNARY_FOLDS = 5

In [10]:
ternary_cv = cross_validate(
    ternary_data,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    n_splits=TERNARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(ternary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(ternary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.516
Fold 2: accuracy=0.547
Fold 3: accuracy=0.535
Fold 4: accuracy=0.535
Fold 5: accuracy=0.542

              precision  recall  f1-score   support
Activation        0.559   0.565     0.562  1208.000
Repression        0.422   0.441     0.431   829.000
None              0.605   0.576     0.590  1018.000
accuracy          0.535   0.535     0.535     0.535
macro avg         0.529   0.527     0.528  3055.000
weighted avg      0.537   0.535     0.536  3055.000


## Save final results

Pickle a `{"binary": CrossValidationResult, "ternary": CrossValidationResult}`
dict to `CV_RESULTS_PATH` for downstream analysis notebooks to load.


In [11]:
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "binary": binary_cv,
    "ternary": ternary_cv,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "gene_subset_path": str(GENE_SUBSET_PATH) if GENE_SUBSET_PATH else None,
}
with open(CV_RESULTS_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved CV results to {CV_RESULTS_PATH}")
print(f"  binary:  lr={binary_cv.config['lr']}, epochs={binary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(binary_cv.fold_accuracies):.3f}")
print(f"  ternary: lr={ternary_cv.config['lr']}, epochs={ternary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(ternary_cv.fold_accuracies):.3f}")


Saved CV results to ../data/results/geneformer_cv_results_nkt.pkl
  binary:  lr=0.001, epochs=8, mean fold acc=0.747
  ternary: lr=0.001, epochs=12, mean fold acc=0.535
